# REXIA-Responsible and Explainable AI


Nicolas Charrondiere

Paul Guimbert

## Partie 1: Données tabulaires

In [5]:
import pandas as pd

df = pd.read_csv('data/RH_dataset.csv', sep=';')
df.head(10)

,Famille d'emploi,Dernière promotion (mois),Dernière augmentation (mois),Début de contrat (années),Ancienneté groupe (années),Etablissement,Âge (années),Parent,Niveau hiérarchique,Salaire (Euros),Statut marital,Véhicule,matricule,label
0,Production,8.510000,7.900000,0.910000,0.970000,27,30,1,1,3199,Marié(e),0,32,0
1,Production,35.119999,22.690001,14.830000,16.299999,7,45,1,2,3861,Marié(e),1,1890,0
2,Production,25.299999,22.139999,17.309999,17.790001,28,49,1,2,4324,PACS,1,1847,0
3,Production,5.240000,5.100000,1.020000,1.750000,27,24,0,1,2641,Célibataire,0,2619,1
4,Production,35.919998,22.840000,8.050000,9.000000,7,46,1,2,5072,Marié(e),1,1963,0
5,Production,30.410000,29.049999,3.350000,4.290000,28,40,1,1,3510,PACS,1,2642,0
6,Commercial/Business,13.840000,32.410000,8.890000,42.509998,28,72,1,3,7223,Marié(e),1,318,0
7,Etudes & Technique,34.799999,22.790001,9.480000,10.580000,28,45,1,2,4132,Marié(e),1,2132,0
8,Etudes & Technique,80.220001,24.180000,8.090000,35.349998,28,65,0,2,4773,Divorcé(e),1,880,0
9,Etudes & Technique,5.190000,2.780000,1.420000,10.100000,28,42,0,2,4517,Union libre,1,676,0


### Analyse du jeu de données

In [22]:
print(f'{len(df.columns)} colonnes : ')
print(*df.columns, sep='\n')

14 colonnes : 
Famille d'emploi
Dernière promotion (mois)
Dernière augmentation (mois)
Début de contrat (années)
Ancienneté groupe (années)
Etablissement
Âge (années)
Parent
Niveau hiérarchique
Salaire (Euros)
Statut marital
Véhicule
matricule
label


14 colonnes:
- 2 variables catégorielles : Famille d'emploi, Statut marital
- 12 variables numériques : Dernière promotion (mois), Dernière augmentation (mois), Début de contrat (années),Ancienneté groupe (années), Etablissement, Âge (années), Parent,Niveau hiérarchique, Salaire (Euros), Véhicule,matricule, label

In [23]:
# Somme des valeurs manquantes par colonne
missing_count = df.isnull().sum()
print(missing_count)

Famille d'emploi                0
Dernière promotion (mois)       0
Dernière augmentation (mois)    0
Début de contrat (années)       0
Ancienneté groupe (années)      0
Etablissement                   0
Âge (années)                    0
Parent                          0
Niveau hiérarchique             0
Salaire (Euros)                 0
Statut marital                  0
Véhicule                        0
matricule                       0
label                           0
dtype: int64


Il n'y a pas de valeur manquantes dans le dataset

In [9]:
df.describe()

,Dernière promotion (mois),Dernière augmentation (mois),Début de contrat (années),Ancienneté groupe (années),Etablissement,Âge (années),Parent,Niveau hiérarchique,Salaire (Euros),Véhicule,matricule,label
count,23857.000000,23857.000000,23857.000000,23857.000000,23857.000000,23857.000000,23857.000000,23857.000000,23857.000000,23857.000000,23857.000000,23857.000000
mean,29.460739,7.934986,7.530322,11.632095,20.193947,41.767154,0.720711,1.554554,4168.404032,0.506853,1361.255858,0.031647
std,25.497874,7.549982,5.985476,9.218618,9.295469,11.014444,0.448659,0.657887,1657.829824,0.500299,794.183153,0.175062
min,0.000000,0.000000,0.000000,0.000000,0.000000,20.000000,0.000000,1.000000,2134.000000,0.000000,0.000000,0.000000
25%,10.590000,3.180000,2.300000,3.800000,11.000000,34.000000,0.000000,1.000000,3197.000000,0.000000,655.000000,0.000000
50%,21.219999,5.880000,6.280000,9.870000,26.000000,41.000000,1.000000,1.000000,3629.000000,1.000000,1371.000000,0.000000
75%,41.400002,10.340000,11.070000,16.320000,28.000000,49.000000,1.000000,2.000000,4511.000000,1.000000,2072.000000,0.000000
max,152.970001,84.050003,33.119999,45.619999,36.000000,100.000000,1.000000,4.000000,18137.000000,2.000000,2675.000000,1.000000


### Analyse des variables sensibles et biais potentiels

In [3]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    # Correction de Yates pour les petits échantillons si nécessaire
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

# Exemple : Lien entre Famille d'emploi et Statut Marital
print(f"V de Cramer : {cramers_v(df['Famille d\'emploi'], df['Statut marital']):.2f}")

V de Cramer : 0.10


In [5]:
def correlation_ratio(categories, values):
    f_obs = values.groupby(categories).apply(list)
    # Calcul de la somme des carrés totale (SST)
    ss_total = np.sum((values - values.mean())**2)
    # Calcul de la somme des carrés inter-groupes (SSB)
    category_means = values.groupby(categories).mean()
    category_counts = values.groupby(categories).count()
    ss_between = np.sum(category_counts * (category_means - values.mean())**2)
    
    return ss_between / ss_total

# Exemple : Est-ce que le Salaire dépend du Statut Marital ?
eta = correlation_ratio(df['Statut marital'], df['Salaire (Euros)'])
print(f"Ratio de corrélation (Eta^2) : {eta:.2f}")

Ratio de corrélation (Eta^2) : 0.11


### Apprentissage automatique

### Partie 2:

### Partie 3: